# Final held-out evaluation, export and replay

Reviewed human FPV baseline; no NVIDIA API calls, cloud runtime creation or robot commands. Set `CAFEROOMBA_REVISION` to the full published implementation commit SHA. Mount persistent storage and set `CAFEROOMBA_WORKSPACE` to the same location in notebooks 01–03. For consumer Colab, explicitly mount Drive if wanted: `from google.colab import drive; drive.mount('/content/drive')`. Enterprise storage/auth varies; use your persistent filesystem. Ordinary `/content` and `/tmp` are not durable across runtime deletion. Never paste keys into cells. See `docs/COLAB_TRAINING.md`.

In [ ]:
import os, re, subprocess, sys, shutil
from pathlib import Path
REPO_REVISION = os.environ.get("CAFEROOMBA_REVISION", "SET_IMPLEMENTATION_COMMIT_SHA")
REPO = Path(os.environ.get("CAFEROOMBA_REPO", "/content/caferoomba"))
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REVISION):
    raise ValueError("Set CAFEROOMBA_REVISION to the full published implementation commit SHA.")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/EdwinKestler/caferoomba.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_REVISION], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual != REPO_REVISION:
    raise ValueError("Existing checkout differs; use a separate checkout without overwriting local work.")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise ValueError("Use a clean checkout for reproducible source provenance.")
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer required.")
# Preserve an existing CUDA-enabled Torch installation; do not force CPU-only wheels.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO) + "[cpu,dev]"], check=True)
sys.path.insert(0, str(REPO / "src"))
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("Install ffmpeg/ffprobe before preparing videos.")
print("source revision:", actual)


In [ ]:
workspace_text = os.environ.get("CAFEROOMBA_WORKSPACE", "")
if not workspace_text:
    raise ValueError("Set CAFEROOMBA_WORKSPACE to your mounted persistent workspace.")
WORKSPACE = Path(workspace_text).resolve()
if not WORKSPACE.is_dir():
    raise ValueError("Workspace must already exist on persistent storage.")
DATASET = WORKSPACE / "dataset-v1"
RUN = WORKSPACE / "run-v1"
MANIFEST = DATASET / "manifest.json"


In [ ]:
from caferoomba.learning.pipeline import evaluate_run, export_run
# Loads notebook 02's validation-selected checkpoint; never trains a new model.
metrics = evaluate_run(MANIFEST, RUN)
print({key: metrics[key] for key in ("n", "accuracy", "macro_f1", "turn180")})
exported = export_run(MANIFEST, RUN, max_samples=16)
print({key: exported[key] for key in ("onnx", "onnx_sha256", "validation_samples", "agrees")})


In [ ]:
from caferoomba.data.prepare import load_prepared_dataset
from caferoomba.learning.dataset import load_clip_stack
from caferoomba.deployment.inference import OnboardPolicy
clips = load_prepared_dataset(MANIFEST)
example = next(c for c in clips if c.split == "val")
policy = OnboardPolicy(RUN / "student.onnx")
prediction = policy.predict(load_clip_stack(example, size=exported["image_size"])[None, ...])
print(prediction)
assert prediction["requires_cloud"] is False


Keep training_report.json, test_metrics.json, export_report.json, best.pt, last.pt, student.onnx and the complete dataset on durable storage. Final evaluation reuses its saved result for the same artifacts. An evaluated run cannot resume. Further experiments require a new run and disclosure of test-set reuse. CPU export is not Orin or driving acceptance.